In [2]:
import folium

# Path to  GeoJSON file
geojson_path = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\ward_vis\mygeodata\London_Ward.geojson"# Replace with your actual path

# Create a Folium map centered on London
m = folium.Map(location=[51.5074, 0.1278], zoom_start=10)

# Add the GeoJSON layer to the map with a tooltip for the 'NAME' field
folium.GeoJson(
    geojson_path,
    tooltip=folium.GeoJsonTooltip(fields=['NAME'], aliases=['Ward Name:'])
).add_to(m)

# Display the map
#m

In [1]:
#bolder with cluster data 
import os
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Paths to your data
clean_data_folder = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\clean"  # Replace with your actual path
geojson_path = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\ward_vis\mygeodata\London_Ward.geojson"# Replace with your actual path
# Create LONDON map
london_map = folium.Map(location=[51.5074, -0.1278], zoom_start=12)

# Add the GeoJSON layer for ward boundaries
folium.GeoJson(
    geojson_path,
    tooltip=folium.GeoJsonTooltip(fields=['NAME'], aliases=['Ward Name:'])  # Add tooltip
).add_to(london_map)


marker_cluster = MarkerCluster().add_to(london_map)

for filename in os.listdir(clean_data_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(clean_data_folder, filename)
        
        df = pd.read_csv(file_path)

        if 'Longitude' in df.columns and 'Latitude' in df.columns:
            # replace "no location" as NaN
            df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
            df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')

            df_clean = df.dropna(subset=['Longitude', 'Latitude'])
            
            # lable of 'Longitude', 'Latitude' added in the MarkerCluster 
            for index, row in df_clean.iterrows():
                folium.Marker([row['Latitude'], row['Longitude']]).add_to(marker_cluster)

#london_map


C:\Users\20231809\.conda\envs\burglary_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
import os
import pandas as pd
import folium
import geopandas as gpd
from folium.plugins import MarkerCluster
from branca.colormap import LinearColormap


clean_data_folder = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\clean"  # Replace with your actual path
geojson_path = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\ward_vis\mygeodata\London_Ward.geojson"# Replace with your actual path

wards_gdf = gpd.read_file(geojson_path)
print("GeoJSON字段:", wards_gdf.columns.tolist())


target_field = 'NAME'
ward_counts = {ward_name: 0 for ward_name in wards_gdf[target_field]}


for filename in os.listdir(clean_data_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(clean_data_folder, filename)
        df = pd.read_csv(file_path)
        
        if {'Longitude', 'Latitude'}.issubset(df.columns):
           
            df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
            df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
            df_clean = df.dropna(subset=['Longitude', 'Latitude'])
            
           
            points_gdf = gpd.GeoDataFrame(
                df_clean,
                geometry=gpd.points_from_xy(df_clean.Longitude, df_clean.Latitude),
                crs=wards_gdf.crs
            )
            
           
            joined = gpd.sjoin(points_gdf, wards_gdf, predicate='within')
            
            
            
            if target_field in joined.columns:
                for ward_name in joined[target_field]:
                    ward_counts[ward_name] += 1
            else:
                print(f"error:  {target_field} is not exist")


max_count = max(ward_counts.values())
colormap = LinearColormap(['#ffffff', '#BF0808'], vmin=0, vmax=max_count)

london_map = folium.Map(location=[51.5074, -0.1278], zoom_start=12)


def style_function(feature):
    count = ward_counts.get(feature['properties'][target_field], 0)
    return {
        'fillColor': colormap(count),
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.6
    }

folium.GeoJson(
    geojson_path,
    name='Ward Heatmap',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=[target_field], aliases=['Ward:'])
).add_to(london_map)

marker_cluster = MarkerCluster().add_to(london_map)
for filename in os.listdir(clean_data_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(clean_data_folder, filename)
        df = pd.read_csv(file_path)
        if {'Longitude', 'Latitude'}.issubset(df.columns):
            df_clean = df.dropna(subset=['Longitude', 'Latitude'])
            for _, row in df_clean.iterrows():
                folium.Marker(
                    [row['Latitude'], row['Longitude']],
                    icon=folium.Icon(icon='circle', prefix='fa', color='darkred', icon_size=(8,8))
                ).add_to(marker_cluster)

colormap.add_to(london_map)
#london_map

GeoJSON字段: ['NAME', 'GSS_CODE', 'DISTRICT', 'LAGSSCODE', 'HECTARES', 'NONLD_AREA', 'geometry']


In [4]:
import os
import pandas as pd
import folium
import geopandas as gpd
from folium.plugins import MarkerCluster
from branca.colormap import LinearColormap


clean_data_folder = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\clean"  # Replace with your actual path
geojson_path = r"C:\Users\20231809\Desktop\study\Y2\Q4\Data Chanllenge\ward_vis\mygeodata\London_Ward.geojson"# 
TARGET_FIELD = 'NAME'  


wards_gdf = gpd.read_file(geojson_path).to_crs(epsg=4326)


ward_counts = {name: 0 for name in wards_gdf[TARGET_FIELD]}
all_points = []

for filename in os.listdir(clean_data_folder):
    if filename.endswith('.csv'):
        df = pd.read_csv(os.path.join(clean_data_folder, filename))
        
        df_clean = df.dropna(subset=['Longitude', 'Latitude']).assign(
            Longitude = lambda x: pd.to_numeric(x['Longitude'], errors='coerce'),
            Latitude = lambda x: pd.to_numeric(x['Latitude'], errors='coerce')
        ).dropna(subset=['Longitude', 'Latitude'])
        
       
        all_points.extend(list(zip(df_clean.Latitude, df_clean.Longitude)))
        
        
        points_gdf = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(df_clean.Longitude, df_clean.Latitude),
            crs=wards_gdf.crs
        )
        joined = gpd.sjoin(points_gdf, wards_gdf, predicate='within')
        for ward in joined[TARGET_FIELD]:
            ward_counts[ward] += 1

colormap = LinearColormap(
    ['#ffffff', '#ffeda0', '#feb24c', '#f03b20'], 
    vmin=0, 
    vmax=max(ward_counts.values()),
    caption='Numbers o'
)

m = folium.Map(location=[51.5074, -0.1278], 
              zoom_start=12,
              tiles='CartoDB positron', 
              control_scale=True)

# graph layer）
folium.GeoJson(
    wards_gdf,
    name='Heatmap',
    style_function=lambda feature: {
        'fillColor': colormap(ward_counts.get(feature['properties'][TARGET_FIELD], 0)),
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[TARGET_FIELD],
        aliases=['wards: '],
        localize=True
    )
).add_to(m)

marker_cluster = MarkerCluster(
    name="cluster",
    options={
        'disableClusteringAtZoom': 15,  # 15 degrees zoom in to show saperate point
        'spiderfyDistanceMultiplier': 2  
    }
).add_to(m)

# add points
for lat, lon in all_points:
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color='#ff0000',
        fill=True,
        fill_opacity=0.6
    ).add_to(marker_cluster)

# layer contral
colormap.add_to(m)
folium.LayerControl().add_to(m)

#m 